# Energy Sentinel — Feature Engineering

This notebook builds features for anomaly detection from cleaned appliance data.
Goals:
- Add time-based features (hour, day, weekend, month)
- Add rolling statistics per appliance (mean, std, max, min)
- Add daily aggregated features (energy, activity rate, cycles)
- Save final feature set to data/processed/

# 1 - Imports & Setup

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.config.settings import settings
from src.features.builder import FeatureBuilder
from src.utils.logger import get_logger

logger = get_logger("feature_engineering")

print("Imports successful")

Imports successful


# 2 - Load Cleaned Data

In [2]:
input_path = settings.PROCESSED_DIR / "house_1_cleaned_2013.parquet"

df_cleaned = pd.read_parquet(input_path)

print(f"Loaded: {input_path}")
print(f"Shape: {df_cleaned.shape}")
print(f"Columns: {list(df_cleaned.columns)}")

Loaded: D:\Github\energy-sentinel\data\processed\house_1_cleaned_2013.parquet
Shape: (525600, 7)
Columns: ['Aggregate', 'Washing Machine', 'Dishwasher', 'TV', 'Kettle', 'Fridge', 'Microwave']


# 3 - Build Features

In [3]:
appliances = ['Aggregate', 'Washing Machine', 'Dishwasher', 'TV', 'Kettle', 'Fridge', 'Microwave']

builder = FeatureBuilder()
df_features = builder.build(df_cleaned, appliances)

print(f"Shape before: {df_cleaned.shape}")
print(f"Shape after : {df_features.shape}")
print(f"\nNew columns added: {len(df_features.columns) - len(df_cleaned.columns)}")
print(f"\nAll columns:\n")
for col in df_features.columns:
    print(f"  {col}")

2026-07-12 06:08:01 | INFO     | src.features.builder | Starting feature engineering pipeline...
2026-07-12 06:08:01 | INFO     | src.features.builder | Time features added: hour, day_of_week, is_weekend, month
2026-07-12 06:08:01 | INFO     | src.features.builder | Aggregate: rolling features added (window=60min)
2026-07-12 06:08:01 | INFO     | src.features.builder | Aggregate: daily features added (energy, activity rate, cycles)
2026-07-12 06:08:01 | INFO     | src.features.builder | Washing Machine: rolling features added (window=60min)
2026-07-12 06:08:01 | INFO     | src.features.builder | Washing Machine: daily features added (energy, activity rate, cycles)
2026-07-12 06:08:02 | INFO     | src.features.builder | Dishwasher: rolling features added (window=60min)
2026-07-12 06:08:02 | INFO     | src.features.builder | Dishwasher: daily features added (energy, activity rate, cycles)
2026-07-12 06:08:02 | INFO     | src.features.builder | TV: rolling features added (window=60min)
20

# 4 - Verify Features

In [4]:
# Sample check — fridge features
print("Fridge feature sample:\n")
fridge_cols = ['Fridge', 'Fridge_rolling_mean', 'Fridge_rolling_std',
               'Fridge_daily_energy_kwh', 'Fridge_daily_activity_rate', 
               'Fridge_daily_cycles']

print(df_features[fridge_cols].head(10).round(3))

# Check for nulls
print(f"\nNull values in features:")
nulls = df_features.isnull().sum()
print(nulls[nulls > 0] if nulls.any() else "No nulls found")

Fridge feature sample:

                              Fridge  Fridge_rolling_mean  Fridge_rolling_std  \
timestamp                                                                       
2013-01-01 00:00:00+00:00  88.800003               88.800               0.000   
2013-01-01 00:01:00+00:00  88.599998               88.700               0.141   
2013-01-01 00:02:00+00:00  88.599998               88.667               0.115   
2013-01-01 00:03:00+00:00  88.889000               88.722               0.146   
2013-01-01 00:04:00+00:00  88.800003               88.738               0.131   
2013-01-01 00:05:00+00:00  88.500000               88.698               0.152   
2013-01-01 00:06:00+00:00  88.699997               88.698               0.139   
2013-01-01 00:07:00+00:00  88.889000               88.722               0.145   
2013-01-01 00:08:00+00:00  88.699997               88.720               0.136   
2013-01-01 00:09:00+00:00  88.599998               88.708               0.134   

   

# 5 - Save Feature Dataset

In [5]:
output_path = settings.PROCESSED_DIR / "house_1_features_2013.parquet"

df_features.to_parquet(output_path)

print(f"Saved: {output_path}")
print(f"Shape: {df_features.shape}")
print(f"Columns: {len(df_features.columns)}")

Saved: D:\Github\energy-sentinel\data\processed\house_1_features_2013.parquet
Shape: (525600, 60)
Columns: 60
